# 09 — Advanced: Observability

**Stage 9 of the workshop (Production, extended).** Tracing every model/tool call with OpenTelemetry, using the console exporter.

## Problem

Getting from "works on my laptop" to something a team can rely on — observability is what lets you see what an agent actually did in production, not just its final text output.

## Concept

Strands emits OTEL spans for every agent invocation, model call, and tool call automatically — you don't instrument your own code, you just attach an exporter. This demo uses the console exporter (prints spans to stdout, no external service, works offline). Shipping real traces to Langfuse instead is the same pattern as `model_provider.get_model()`: same agent code, only the exporter/endpoint changes.

```python
import os
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://cloud.langfuse.com/api/public/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {base64_encoded_key}"
StrandsTelemetry().setup_otlp_exporter()
```

## Architecture

```
StrandsTelemetry().setup_console_exporter()   ← attached once, before any Agent runs
        │
        ▼
agent("What is 12 plus 30? Use the add tool.")
        │
        ├─ span: agent invocation
        ├─ span: model call
        ├─ span: tool call (add)
        ├─ span: model call (final answer)
        ▼
   spans printed to stdout (console exporter)
        │
        ▼
   agent's normal text result
```

## Step 1 — Attach the console exporter (before model/agent setup)

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent, tool
from strands.telemetry import StrandsTelemetry

StrandsTelemetry().setup_console_exporter()

model = get_model()


{
    "name": "chat",
    "context": {
        "trace_id": "0x1cdd22c172748ca7b11907d8b44dfc64",
        "span_id": "0x55c4765257d1de67",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6b87b074b1ce6eac",
    "start_time": "2026-09-25T13:51:16.728737Z",
    "end_time": "2026-09-25T13:51:17.330945Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-09-25T13:51:16.728738+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "gen_ai.request.model": "qwen3.5:4b",
        "gen_ai.event.end_time": "2026-09-25T13:51:17.330915+00:00",
        "gen_ai.usage.prompt_tokens": 13,
        "gen_ai.usage.input_tokens": 13,
        "gen_ai.usage.completion_tokens": 2,
        "gen_ai.usage.output_tokens": 2,
        "gen_ai.usage.total_tokens": 15,
        "gen_ai.server.time_to_first_token": 15,
        "gen_ai.server.request.duration": 299
    },
    "e

{
    "name": "execute_event_loop_cycle",
    "context": {
        "trace_id": "0x1cdd22c172748ca7b11907d8b44dfc64",
        "span_id": "0x6b87b074b1ce6eac",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x97dff82cf3605fef",
    "start_time": "2026-09-25T13:51:16.728545Z",
    "end_time": "2026-09-25T13:51:17.334317Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-09-25T13:51:16.728548+00:00",
        "gen_ai.operation.name": "execute_event_loop_cycle",
        "gen_ai.system": "strands-agents",
        "event_loop.cycle_id": "38454ec6-0681-457f-9c7e-26ad1733b79d",
        "gen_ai.event.end_time": "2026-09-25T13:51:17.334294+00:00"
    },
    "events": [
        {
            "name": "gen_ai.user.message",
            "timestamp": "2026-09-25T13:51:16.728586Z",
            "attributes": {
                "content": "[{\"text\": \"ping\"}]"
            }
        }
    ],
    "links"

{
    "name": "invoke_agent Strands Agents",
    "context": {
        "trace_id": "0x1cdd22c172748ca7b11907d8b44dfc64",
        "span_id": "0x97dff82cf3605fef",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-09-25T13:51:16.728238Z",
    "end_time": "2026-09-25T13:51:17.335258Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-09-25T13:51:16.728244+00:00",
        "gen_ai.operation.name": "invoke_agent",
        "gen_ai.system": "strands-agents",
        "gen_ai.agent.name": "Strands Agents",
        "gen_ai.request.model": "qwen3.5:4b",
        "gen_ai.event.end_time": "2026-09-25T13:51:17.335232+00:00",
        "gen_ai.usage.prompt_tokens": 13,
        "gen_ai.usage.completion_tokens": 2,
        "gen_ai.usage.input_tokens": 13,
        "gen_ai.usage.output_tokens": 2,
        "gen_ai.usage.total_tokens": 15,
        "gen_ai.usage.cache_read.input_tokens"

## Step 2 — Define the tool and agent

`trace_attributes` attaches custom metadata to every span this agent produces.

In [2]:
@tool
def add(x: int, y: int) -> int:
    """Add two numbers."""
    return x + y


agent = Agent(model=model, tools=[add], trace_attributes={"workshop.module": "09-advanced"})


## Step 3 — Run it

OTEL spans print to stdout above the `---` separator; the agent's normal result prints after.

In [3]:
result = agent("What is 12 plus 30? Use the add tool.")
print("---")
print(result)



Tool #1: add
{
    "name": "chat",
    "context": {
        "trace_id": "0x6807131a53d1e180df9e20be70e87105",
        "span_id": "0x49fdd35a15ad8f2d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x192308b625ecbf10",
    "start_time": "2026-09-25T13:51:17.355057Z",
    "end_time": "2026-09-25T13:51:18.563324Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-09-25T13:51:17.355059+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "workshop.module": "09-advanced",
        "gen_ai.request.model": "qwen3.5:4b",
        "gen_ai.event.end_time": "2026-09-25T13:51:18.563290+00:00",
        "gen_ai.usage.prompt_tokens": 306,
        "gen_ai.usage.input_tokens": 306,
        "gen_ai.usage.completion_tokens": 37,
        "gen_ai.usage.output_tokens": 37,
        "gen_ai.usage.total_tokens": 343,
        "gen_ai.server.time_to_first_token": 20

{
    "name": "execute_tool add",
    "context": {
        "trace_id": "0x6807131a53d1e180df9e20be70e87105",
        "span_id": "0x44f69d1bdf62747e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x192308b625ecbf10",
    "start_time": "2026-09-25T13:51:18.564577Z",
    "end_time": "2026-09-25T13:51:18.565417Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-09-25T13:51:18.564579+00:00",
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.system": "strands-agents",
        "gen_ai.tool.name": "add",
        "gen_ai.tool.call.id": "tooluse_59d00cc08d734bd8add96de0",
        "workshop.module": "09-advanced",
        "gen_ai.tool.description": "Add two numbers.",
        "gen_ai.tool.json_schema": "{\"properties\": {\"x\": {\"description\": \"Parameter x\", \"type\": \"integer\"}, \"y\": {\"description\": \"Parameter y\", \"type\": \"integer\"}}, \"required\": [\"x\", \"y\"

{
    "name": "execute_event_loop_cycle",
    "context": {
        "trace_id": "0x6807131a53d1e180df9e20be70e87105",
        "span_id": "0x192308b625ecbf10",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x679f8b66fab0f8a5",
    "start_time": "2026-09-25T13:51:17.354561Z",
    "end_time": "2026-09-25T13:51:18.566422Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-09-25T13:51:17.354565+00:00",
        "gen_ai.operation.name": "execute_event_loop_cycle",
        "gen_ai.system": "strands-agents",
        "event_loop.cycle_id": "7299cf59-1e52-4cb7-9a6c-014c51f006ad",
        "workshop.module": "09-advanced",
        "gen_ai.event.end_time": "2026-09-25T13:51:18.566409+00:00"
    },
    "events": [
        {
            "name": "gen_ai.user.message",
            "timestamp": "2026-09-25T13:51:17.354672Z",
            "attributes": {
                "content": "[{\"text\": \"What is 12

The result of 12 plus 30 is 42

.{
    "name": "chat",
    "context": {
        "trace_id": "0x6807131a53d1e180df9e20be70e87105",
        "span_id": "0x95f450370bd9143e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x4fcdb515b5ff0f7b",
    "start_time": "2026-09-25T13:51:18.567643Z",
    "end_time": "2026-09-25T13:51:19.173096Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-09-25T13:51:18.567645+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "workshop.module": "09-advanced",
        "gen_ai.request.model": "qwen3.5:4b",
        "gen_ai.event.end_time": "2026-09-25T13:51:19.173068+00:00",
        "gen_ai.usage.prompt_tokens": 358,
        "gen_ai.usage.input_tokens": 358,
        "gen_ai.usage.completion_tokens": 16,
        "gen_ai.usage.output_tokens": 16,
        "gen_ai.usage.total_tokens": 374,
        "gen_ai.server.time_to_first_token": 17,
        "ge

{
    "name": "execute_event_loop_cycle",
    "context": {
        "trace_id": "0x6807131a53d1e180df9e20be70e87105",
        "span_id": "0x4fcdb515b5ff0f7b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x679f8b66fab0f8a5",
    "start_time": "2026-09-25T13:51:18.567341Z",
    "end_time": "2026-09-25T13:51:19.174022Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-09-25T13:51:18.567344+00:00",
        "gen_ai.operation.name": "execute_event_loop_cycle",
        "gen_ai.system": "strands-agents",
        "event_loop.cycle_id": "3f836632-9669-4bc5-948a-f3ed1a5b3fff",
        "workshop.module": "09-advanced",
        "event_loop.parent_cycle_id": "7299cf59-1e52-4cb7-9a6c-014c51f006ad",
        "gen_ai.event.end_time": "2026-09-25T13:51:19.174005+00:00"
    },
    "events": [
        {
            "name": "gen_ai.user.message",
            "timestamp": "2026-09-25T13:51:18.567389Z",
  

{
    "name": "invoke_agent Strands Agents",
    "context": {
        "trace_id": "0x6807131a53d1e180df9e20be70e87105",
        "span_id": "0x679f8b66fab0f8a5",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-09-25T13:51:17.354112Z",
    "end_time": "2026-09-25T13:51:19.174959Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-09-25T13:51:17.354117+00:00",
        "gen_ai.operation.name": "invoke_agent",
        "gen_ai.system": "strands-agents",
        "gen_ai.agent.name": "Strands Agents",
        "gen_ai.request.model": "qwen3.5:4b",
        "gen_ai.agent.tools": "[\"add\"]",
        "workshop.module": "09-advanced",
        "gen_ai.event.end_time": "2026-09-25T13:51:19.174932+00:00",
        "gen_ai.usage.prompt_tokens": 664,
        "gen_ai.usage.completion_tokens": 53,
        "gen_ai.usage.input_tokens": 664,
        "gen_ai.usage.output_tokens": 53

---
The result of 12 plus 30 is 42.

